# EDA Current Stress

Template eksperimen Current Stress dengan tracking MLflow yang konsisten.

In [ ]:
from pathlib import Path
import tempfile
import mlflow
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

from mlflow_utils import (
    DATASET_NAME,
    DATASET_VERSION,
    EXPERIMENT_NAME,
    RANDOM_STATE,
    TEST_SIZE,
    build_preprocessor,
    evaluate_classification,
    load_dataset,
    log_and_register_model,
    log_classification_artifacts,
    log_dataset_inputs,
    log_run_metadata,
    select_features,
    set_seeds,
    setup_mlflow,
)


In [ ]:
# 1) Header & Config
set_seeds(RANDOM_STATE)
setup_mlflow()

# 2) Load dataset + basic cleaning
raw_df, X, y = load_dataset()
X = select_features(X, feature_group="all")

# 3) Split train/test (untuk dokumentasi dataset input)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# 4) MLflow logging
run_name = "EDA Current Stress"
description = (
    "EDA dataset current stress; features=all; dataset=current_stress_v1; "
    "split=80/20; random_state=42; no model training"
)
with mlflow.start_run(run_name=run_name) as run:
    tags = {
        "project": "nostressia",
        "task": "current-stress",
        "features": "all",
        "model": "EDA",
        "dataset_name": DATASET_NAME,
        "dataset_version": DATASET_VERSION,
        "split": "80/20",
        "random_state": str(RANDOM_STATE),
    }
    log_run_metadata(tags, description)
    mlflow.log_params({"notebook": "01_eda_current_stress", "rows": len(raw_df), "cols": raw_df.shape[1]})
    log_dataset_inputs(X_train, y_train, X_test, y_test)

    with tempfile.TemporaryDirectory() as td:
        artifact_dir = Path(td)
        raw_df.describe(include="all").transpose().to_csv(artifact_dir / "eda_describe.csv")
        raw_df.isna().sum().rename("missing_count").to_csv(artifact_dir / "eda_missing.csv")
        raw_df.head(100).to_csv(artifact_dir / "eda_sample.csv", index=False)
        mlflow.log_artifacts(str(artifact_dir), artifact_path="eda")

print(f"Run selesai: {run.info.run_id}")
